# Lexos Word Cloud Tutorial

A word cloud is a visual representation of text data, where the size of each word indicates its frequency or importance within the text. In Lexos, you can create word clouds using the `WordCloud` class from the `lexos.visualization.cloud` module. This tutorial will guide you through generating a word cloud from your documents. We'll cover both single word clouds and multiple word clouds (multiclouds) with various data input types and customization options.

The simplest way to generate a wordcloud in Lexos is to import the `wordcloud` function and pass it some raw text.

In [ ]:
# Import the wordcloud function from lexos.visualization.cloud
from lexos.visualization.cloud import wordcloud

# Sample text for generating a word cloud
sample_text = """
Natural language processing is a fascinating field that combines linguistics, computer science, and artificial intelligence. Text analysis, sentiment analysis, and language modeling are key components of modern NLP systems. Machine learning algorithms help computers understand and process human language effectively.
"""

# Create a basic word cloud
wordcloud(sample_text)


If you provide a filename in the `path` parameter, the word cloud will be saved to a file (in `.png` or `jpg` format, depending the file extension you use).

Calling `wordcloud(..., show=False)` returns the raw `WordCloud` object.

You can then inspect attributes like `.width` (pixel width), `.height` (pixel height), and `.words_` (a dict of word-to-frequency mapping).  By using the `WordCloud` object's internal methods, it is also possible to save the the image in SVG format with the following code:

```python
wc = wordcloud(..., show=False)
wc.to_svg("filename.svg")
```

See the [`WordCloud` API documentation](https://amueller.github.io/word_cloud/generated/wordcloud.WordCloud.html) for a full description of the `WordCloud` class.

One feature that is not well documented there is the `to_image` method. You can use this to display a word cloud which you have saved to a variable.

```
wc.to_image().show()
```

In [ ]:
wc = wordcloud(sample_text, show=False)
wc.to_image().show()

You can also use a dictionary of term frequencies, where the keys are terms and the values are counts:

In [ ]:
# Sample word frequency data
word_frequencies = {
    "machine": 25,
    "learning": 20,
    "artificial": 18,
    "intelligence": 15,
    "data": 22,
    "science": 19,
    "algorithm": 12,
    "neural": 10,
    "network": 14,
    "python": 16,
    "analysis": 13,
    "model": 17
}

# Create word cloud from frequencies
wordcloud(word_frequencies)

Similarly, you can use a pandas dataframe in which the terms are rows and the documents are columns.

In [ ]:
import pandas as pd

terms = ["machine", "learning", "data", "science", "python", "analysis", "model", "algorithm"]
doc_data = {
    "doc1": [15, 12, 20, 8, 10, 6, 9, 7],
    "doc2": [8, 15, 18, 12, 14, 10, 11, 5],
    "doc3": [12, 10, 15, 14, 8, 12, 13, 9]
}

df = pd.DataFrame(doc_data, index=terms)
print("Sample DataFrame:")
display(df)

# Create word cloud from DataFrame (combines all documents)
wordcloud(df)

Lexos also supports word cloud generation from a list containing lists of strings, where each inner list represents a document as a list of tokens.

This format is common in pre-tokenized datasets or NLP pipelines. Here's a basic example with three tokenized documents.

In [ ]:
docs_as_tokens = [
    ["lexos", "visualizes", "text"],
    ["word", "cloud", "generation"],
    ["tokenized", "input", "is", "supported"]
]

wordcloud(docs_as_tokens, docs=0)  # Visualize just the first document


### Word Cloud from a spaCy Doc

The Python `WordCloud` package will automatically tokenize raw text using white space, but we may want to take advantage of a language model instead, especially as this can allow us to do some of our own filtering. For this, we can leverage the power of spaCy Doc objects. The following example shows how we can load a text, convert it to a spaCy Doc, and then perform some filtering before submitting it to the `wordcloud` function.

Since our sample text is a long novel, we will take a snippet consisting of the first 2000 characters.

When we filter the tokens, we'll perform some of the more common steps: removing punctuation, digits, and stop words. This can help to produce more meaningful word clouds.

In [ ]:
# Import the Lexos Loader and Tokenizer classes
from lexos.io.loader import Loader
from lexos.tokenizer import Tokenizer

# Load and slice real document
loader = Loader()
loader.load(paths=["docs/Austen_Pride.txt"])

# Grab the first 2000 characters of the text
snippet = loader.texts[0][:2000]

# Tokenize the snippet to create a document object
tokenizer = Tokenizer(model="en_core_web_sm")
doc = tokenizer.make_doc(snippet)

# Filter tokens to remove unwanted types
tokens = [
    token.lower_
    for token in doc
    if not token.is_punct
    and not token.is_space
    and token.is_alpha
    and not token.is_stop
]

# Now plot using the list of tokens
wordcloud(tokens)


## Styling Options

- `sample_text` (str): The same multiline NLP description string from the previous cell.
- `custom_opts` (dict): A dictionary specifying styling parameters for the word cloud:

  - `"background_color"`: Color of the plot background ("black").
  - `"colormap"`: Color scheme used for words ("viridis").
  - `"max_words"`: Maximum number of words to display (100).
  - `"contour_color"` / `"contour_width"`: Draws a contour around the cloud in white with width 1.
  - `"width"` / `"height"`: Pixel dimensions of the word cloud (800×400).
- `figure_opts` (dict): A dictionary for Matplotlib figure settings; here, `{"figsize": (12, 6)}` sets an overall figure size of 12×6 inches.

In [ ]:
# Custom styling options
custom_opts = {
    "background_color": "black",
    "colormap": "viridis",
    "max_words": 100,
    "contour_width": 2,
    "contour_color": "white",
    "width": 800,
    "height": 400
}

# Create a styled word cloud
wordcloud(
    sample_text,
    opts=custom_opts,
    figure_opts={"figsize": (12, 6)}
)


## Creating a Circular Word Cloud

The `round` argument instructs the underlying `wordcloud` generator to use a circular mask of the specified radius. By combining that with styling options, you get a neat circular arrangement of words.

In [ ]:
# Create a circular word cloud
wordcloud(
    sample_text,
    opts={
        "background_color": "white",
        "colormap": "cool",
        "max_words": 150
    },
    round=120,  # Controls the roundness (100-300 works well)
    figure_opts={"figsize": (8, 8)}
)

### Multicloud Visualization (Grid of Word Clouds)

Lexos provides a `multicloud()` function that lets you visualize multiple documents as a grid of word clouds. This is especially useful when comparing text patterns across different documents.

The `multicloud()` function accepts the same input types as `wordcloud()` (including DTM, list of strings, etc.), and supports additional options such as:
- `ncols`: number of columns in the grid
- `labels`: list of document titles
- `title`: overall figure title
- `show`: display or return word clouds
- `round`: optional mask for circular clouds

Below, we visualize 4 short text snippets using this function.


In [ ]:
from lexos.visualization.cloud import multicloud

# Sample multiple documents
docs = [
    "Lexos helps analyze literary texts and visualize patterns.",
    "This is a simple word cloud from a basic string.",
    "Multiple documents can be compared using multicloud.",
    "Token frequency reveals insights about word usage."
]

# Plot multicloud
multicloud(
    data=docs,
    ncols=2,
    labels=["Doc 1", "Doc 2", "Doc 3", "Doc 4"],
    title="Grid of Word Clouds",
    round=120,
    figure_opts={"figsize": (10, 6)}
)


The example below shows how we might split a spaCy document into segments for comparison.

In [ ]:
# Import the Lexos Loader and Tokenizer classes
from lexos.io.loader import Loader
from lexos.tokenizer import Tokenizer

# Load and slice real document
loader = Loader()
loader.load(paths=["docs/Austen_Pride.txt"])

# Split into 3 chunks
text = loader.texts[0]
segments = [text[:3000], text[3000:6000], text[6000:9000]]

# Tokenize the snippet to create a document object
tokenizer = Tokenizer(model="en_core_web_sm")
segments = tokenizer.make_docs(segments)

# Filter tokens to remove unwanted types
token_lists = []
for chunk in segments:
    tokens = [
        token.lower_
        for token in chunk
        if not token.is_stop
        and not token.is_punct
        and not token.is_space
        and token.is_alpha
    ]
    token_lists.append(tokens)

# Pass token lists directly into multicloud
multicloud(
    data=token_lists,
    labels=["Beginning", "Middle", "End"],
    ncols=3,
    round=120,
    title="Pride and Prejudice: Word Cloud by Section",
    figure_opts={"figsize": (15, 5)}
)

## Plotly Word Clouds



## Supported Input Types

The `plotly_wordcloud()` function accepts a wide range of data formats, which makes it flexible across NLP pipelines and document processing tasks. Internally, the function detects the input type and processes it accordingly.

### Acceptable formats for `data`:

| Input Type                  | Description |
|----------------------------|-------------|
| `str`                      | A raw text string |
| `list[str]`                | A list of tokenized words |
| `dict[str, int]`           | A dictionary of word frequencies |
| `spacy.tokens.Doc`         | A spaCy document object |
| `spacy.tokens.Span`        | A span from a spaCy document |
| `list[Doc]`, `list[Span]`  | A list of spaCy Docs or Spans |
| `list[list[str]]`          | A list of tokenized documents (as lists of words) |
| `list[Token]`              | A list of spaCy Token objects |
| `list[list[Token]]`        | A list of lists of Tokens |
| `pandas.DataFrame`         | A term-document matrix (terms as index, docs as columns) |
| `lexos.dtm.DTM`            | A Lexos document-term matrix object |

---

### How the function handles each:

- If you pass **text**, it calls `WordCloud.generate_from_text(text)`
- If you pass a **Doc or Span**, it uses a frequency counter over `token.text`
- If you pass **dict, DataFrame, or DTM**, it uses `generate_from_frequencies(...)`
- For **list-based inputs**, helper functions in `lexos.visualization.processors` standardize the format

> 🔍 The input format must be consistent. Mixed-type lists (e.g. `["word", Doc, ["more words"]]`) will raise a `LexosException`.

---

In the upcoming cell, we'll use a `.txt` file as our input and generate the word cloud directly from raw text.


In [ ]:
# Import the Plotly word cloud function
from lexos.visualization.plotly_wordcloud import plotly_wordcloud

# Generate a basic word cloud using the raw text

fig = plotly_wordcloud(text)

# Note:
# - This will use default WordCloud settings (white background, 2000 max words)
# - It will automatically open a Plotly figure in your notebook or browser (if supported)


## Interpreting the Word Cloud Output

After calling `plotly_wordcloud()` with a valid input, an interactive word cloud is generated using Plotly. Here's how to interpret the visualization:

- **Word Size**: Represents the relative frequency of each word. Larger words appear more often in the input data.
- **Color**: Each word is randomly assigned a color by the `WordCloud` generator. Colors can help visually separate frequent terms.
- **Position**: Words are arranged algorithmically to maximize space usage while avoiding overlap.
- **Hover Info**: Hovering over a word displays its relative frequency (e.g., `"data: 12.50%"`).

This visualization provides an intuitive way to explore prominent terms in your dataset.

> Note: The layout does not include axes or grid lines, as they are intentionally hidden to maintain focus on the words.



---


##  Customizing the Word Cloud: Optional Arguments

The `plotly_wordcloud()` function offers several optional parameters to help you tailor the output to your needs:

---

###  `opts`: WordCloud Options (passed to `WordCloud()`)

Customize the appearance of the word cloud using a dictionary of options.  
These are passed directly to the [`wordcloud.WordCloud`](https://amueller.github.io/word_cloud/generated/wordcloud.WordCloud.html) constructor.

**Example:**
```python
opts = {
    "background_color": "white",
    "max_words": 150,
    "contour_width": 2,
    "contour_color": "navy"
}
````

---

### `layout`: Plotly Layout Customization

You can adjust the layout of the Plotly figure (e.g., width, height, margins, background).
Pass a dictionary of options that updates the default Plotly layout.

**Example:**

```python
layout = {
    "width": 900,
    "height": 600,
    "margin": {"l": 40, "r": 40, "t": 80, "b": 40}
}
```

---

### `path`: Save to File

If you pass a file path (as a string or `Path` object), the function will save the word cloud as an interactive `.html` file.

**Example:**

```python
path = "output/wordcloud.html"
```

---

### `show`: Toggle Display

Controls whether the word cloud is shown immediately or returned as a `go.Figure` object for further use.

* `True` (default): calls `fig.show()`
* `False`: returns the Plotly figure without displaying it

---

### `docs`: Document Selection (for DTM or DataFrame)

Use this to specify one or more documents from a DTM or DataFrame.

* Can be an `int`, `str`, or list of either
* Ignored for other input types

**Example:**

```python
docs = [0, 1]  # Selects the first and second columns of a DTM
```

---

> These optional arguments allow you to fine-tune visual style, selectively generate clouds, or export for later use.



In [ ]:
# Customized Word Cloud: Appearance + Save to File

# Define custom WordCloud options
opts = {
    "background_color": "white",
    "max_words": 100,
    "contour_width": 1,
    "contour_color": "darkred"
}

# Define custom Plotly layout
layout = {
    "width": 425,
    "height": 300,
    "margin": {"l": 30, "r": 30, "t": 60, "b": 30},
    "title": {"text": "Custom Word Cloud", "x": 0.5, "xanchor": "center"}
}

# Define output path and ensure the directory exists
# path = Path("output/custom_wordcloud.html")
# path.parent.mkdir(parents=True, exist_ok=True)
path = None

# Generate and save the word cloud
text = loader.texts[0]  # Use the text above
fig = plotly_wordcloud(text, opts=opts, layout=layout, path=path, show=True)
